<a href="https://colab.research.google.com/github/Agac-ac/FUNDAI-Laboratories-Agac-ac/blob/main/Lab4_Logic_KR_Agac_ac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Logic and Knowledge Representation

## Fundamentals of Artificial Intelligence

**Name:** Jhonel L. Agac-ac
**Course:** BSCS AI
**Section:**  09282-FUNDAI
**Date:** 09/15/26

**GitHub URL:** https://github.com/Agac-ac/FUNDAI-Laboratories-Agac-ac

## Description
This laboratory uses Python and SymPy to perform truth table generation, satisfiability checking, theorem proving, and logical deduction.

In [ ]:
from sympy import symbols, And, Or, Not, Implies, Equivalent, satisfiable
from itertools import product

P, Q, R = symbols('P Q R')

def print_truth_table(expression, symbol_list):
    header = [str(s) for s in symbol_list] + [str(expression)]
    print(" | ".join(header))
    print("-" * (5 * len(header)))

    for values in product([False, True], repeat=len(symbol_list)):
        mapping = dict (zip(symbol_list, values))
        result = bool(expression.subs(mapping))
        row = [str(v) for v in values] + [str(result)]
        print(" | ".join(row))

    print()

In [ ]:
print("Negation: NOT P")
print_truth_table(Not(P), [P])

print("Conjunction: P AND Q")
print_truth_table(And(P, Q), [P, Q])

print("Disjunction: P OR Q")
print_truth_table(Or(P, Q), [P, Q])

print("Implication: P -> Q")
print_truth_table(Implies(P, Q), [P, Q])

print("Biconditional: P <-> Q")
print_truth_table(Equivalent(P, Q), [P, Q])

Negation: NOT P
P | ~P
----------
False | True
True | False

Conjunction: P AND Q
P | Q | P & Q
---------------
False | False | False
False | True | False
True | False | False
True | True | True

Disjunction: P OR Q
P | Q | P | Q
---------------
False | False | False
False | True | True
True | False | True
True | True | True

Implication: P -> Q
P | Q | Implies(P, Q)
---------------
False | False | True
False | True | True
True | False | False
True | True | True

Biconditional: P <-> Q
P | Q | Equivalent(P, Q)
---------------
False | False | True
False | True | False
True | False | False
True | True | True



In [ ]:
def is_tautology(expression, symbol_list):
    for values in product([False, True], repeat=len(symbol_list)):
        mapping = dict(zip(symbol_list, values))
        if not bool(expression.subs(mapping)):
            return False
    return True

law_of_excluded_middle = Or(P, Not(P))
contradiction = And(P, Not(P))
simple_implication = Implies(P, Q)

print("P OR NOT P is a tautology:", is_tautology(law_of_excluded_middle, [P]))
print("P AND NOT P is a tautology:", is_tautology(contradiction, [P]))
print("P -> Q is a tautology:", is_tautology(simple_implication, [P, Q]))

P OR NOT P is a tautology: True
P AND NOT P is a tautology: False
P -> Q is a tautology: False


In [ ]:
def is_satisfiable(expression):
    return satisfiable(expression) is not False

print("P AND NOT P is satisfiable:", is_satisfiable(And(P, Not(P))))
print("P OR Q is satisfiable:", is_satisfiable(Or(P, Q)))
print("P -> Q is satisfiable:", is_satisfiable(Implies(P, Q)))

P AND NOT P is satisfiable: False
P OR Q is satisfiable: True
P -> Q is satisfiable: True


In [ ]:
def to_conjunction(kb):
    if isinstance(kb, list):
        return And(*kb)
    return kb


def kb_entails(kb, conclusion):
    kb_expression = to_conjunction(kb)
    counter_check = And(kb_expression, Not(conclusion))
    return satisfiable(counter_check) is False


def check_entailment(kb, conclusion, label="Query"):
    holds = kb_entails(kb, conclusion)
    kb_expression = to_conjunction(kb)
    counterexample = satisfiable(And(kb_expression, Not(conclusion)))
    print(label)

    if holds:
        print("Result: Entailment holds.")
    else:
        print("Result: Entailment does not hold.")
        print("Counterexample model:", counterexample)

    print("-" * 60)
    return holds

In [ ]:
Rain, Wet = symbols('Rain Wet')

kb_rain = [
    Implies(Rain, Wet),
    Rain
]

check_entailment(kb_rain, Wet, "Theorem Proving: Rain example")

Theorem Proving: Rain example
Result: Entailment holds.
------------------------------------------------------------


True

In [ ]:
kb_invalid = [
    Implies(Rain, Wet),
    Wet
]

check_entailment(kb_invalid, Rain, "Invalid Inference: Affirming the consequent")

Invalid Inference: Affirming the consequent
Result: Entailment does not hold.
Counterexample model: {Wet: True, Rain: False}
------------------------------------------------------------


False

In [ ]:
print("Logical Deduction Rules")
print("=" * 60)

# Modus Ponens
check_entailment(
    [P, Implies(P, Q)],
    Q,
    "Modus Ponens: P, P -> Q, therefore Q"
)

# Modus Tollens
check_entailment(
    [Not(Q), Implies(P, Q)],
    Not(P),
    "Modus Tollens: NOT Q, P -> Q, therefore NOT P"
)

Logical Deduction Rules
Modus Ponens: P, P -> Q, therefore Q
Result: Entailment holds.
------------------------------------------------------------
Modus Tollens: NOT Q, P -> Q, therefore NOT P
Result: Entailment holds.
------------------------------------------------------------


True

## Grounded First-Order Logic Example

Full First-Order Logic includes objects and quantifiers.
For this laboratory, we demonstrate a simple grounded FOL example
by converting FOL atoms into propositional symbols.

English:
- All humans are mortal.
- Socrates is human.
- Therefore, Socrates is mortal.

Grounded propositional form:
- Human_Socrates -> Mortal_Socrates

In [ ]:
Human_Socrates, Mortal_Socrates = symbols('Human_Socrates Mortal_Socrates')

kb_socrates = [
    Implies(Human_Socrates, Mortal_Socrates),
    Human_Socrates
]

check_entailment(
    kb_socrates,
    Mortal_Socrates,
    "Grounded FOL: Socrates is mortal"
)

Grounded FOL: Socrates is mortal
Result: Entailment holds.
------------------------------------------------------------


True

In [ ]:
def make_human_mortal_kb(constants):
    kb = []
    human = {}
    mortal = {}

    for name in constants:
        h, m = symbols(f'Human_{name} Mortal_{name}')
        human[name] = h
        mortal[name] = m
        kb.append(Implies(h, m))

    return kb, human, mortal


constants = ["Socrates", "Plato"]

kb_people, human, mortal = make_human_mortal_kb(constants)

# Add facts
kb_people.append(human["Socrates"])
kb_people.append(human["Plato"])

# Query: Is Plato mortal?
check_entailment(
    kb_people,
    mortal["Plato"],
    "Grounded FOL with multiple constants: Is Plato mortal?"
)

Grounded FOL with multiple constants: Is Plato mortal?
Result: Entailment holds.
------------------------------------------------------------


True

## Guide Questions and Answers

### 1. What is the difference between syntax and semantics?
**Answer:** Syntax is just the grammar rules  what counts as a "legal" sentence in the logic like knowing P ∧ Q is written correctly, but P ∧ ∧ Q is gibberish. Semantics is the meaning  it tells you what the sentence is actually saying and whether it's true or false. Think of it like: syntax = can you spell the word right, semantics = do you know what the word means.
### 2. Why is `P -> Q` true when `P` is false?
**Answer:** P → Q basically means "if P happens, then Q happens." It's only considered a lie false when P actually happens but Q doesn't. If P never happens in the first place, you can't really call it a broken promise  so we just say it's true by default. Example: "If it rains, I'll bring an umbrella." If it doesn't rain, you can't accuse me of lying, whether or not I brought an umbrella.

### 3. What does it mean for a knowledge base to entail a conclusion?
**Answer:** It means the conclusion has to be true, given everything in the knowledge base  no exceptions, no wiggle room. In every single scenario where all the KB's facts are true, the conclusion is also true. It's not a "probably"  it's a guarantee.

### 4. How does theorem proving use satisfiability checking?
**Answer:** Instead of trying to directly prove something is true, we try a sneaky trick: assume the opposite the conclusion is false and add that to the knowledge base. Then we check if this combo can even be satisfied. If it's impossible  a contradiction  that means our original conclusion must be true after all. So proving something true = showing that assuming it's false leads to a contradiction

### 5. What is one limitation of propositional logic compared to First-Order Logic?
**Answer:** Propositional logic can't talk about objects or general rules  it just deals with whole statements as one chunk. So if you wanted to say "all humans are mortal," propositional logic can't say that in one line  you'd have to write out "Socrates is mortal," "Plato is mortal," etc., for every single person, one by one. First-Order Logic fixes this by letting you use variables and quantifiers like "for all x" so you can make one general statement that covers everything.

## Reflection

### Challenges Encountered
- Honestly, the trickiest part for me wasn't the logic itself — it was getting used to how SymPy represents things. At first I kept expecting Implies(P, Q) to just print as True or False right away, but I had to remember it's a symbolic expression until you actually .subs() in real values. Building the print_truth_table() function also took a bit of trial and error, especially looping through all combinations of True/False with product() and making sure the mapping between symbols and values lined up correctly.

The part that really made me pause was kb_entails(). The idea of proving something is true by trying to prove the opposite is impossible felt backwards at first — like, why not just check if the KB directly implies the conclusion? But once I saw it working with the rain/wet example and the "affirming the consequent" example (where it correctly said entailment does not hold and even gave me a counterexample model), it clicked. That counterexample output was actually really helpful for debugging my own understanding of why something wasn't a valid inference.

Scaling up to multiple people (Socrates, Plato) with make_human_mortal_kb() was also a bit of a challenge conceptually — I had to think through why we couldn't just write one generic "Human(x) → Mortal(x)" rule the way real First-Order Logic does, and instead had to generate a separate implication for every single constant. It made the limitation of propositional logic feel very real instead of just something abstract from the readings.

### What I Learned
- This lab really helped me see the difference between syntax and semantics in a hands-on way  writing the logical expressions was the syntax part, and running them through truth tables/satisfiability checks was where the actual meaning showed up. I also finally have an intuitive grip on why P -> Q is true whenever P is false; seeing it play out row-by-row in the truth table made it stick way better than just memorizing the rule.

The biggest takeaway for me was how theorem proving is really just satisfiability checking in disguise  negate the conclusion, throw it in with the KB, and see if it blows up into a contradiction. That "proof by refutation" trick is something I hadn't fully appreciated before this lab, and coding it out made it way more concrete than just reading about it.

Lastly, working through the grounded FOL example with Socrates and Plato made the gap between propositional logic and First-Order Logic really obvious  I could literally see how much repetitive code it takes to represent something that FOL could say in one clean sentence with a quantifier. That's probably the lesson that'll stick with me most going into the next lab.